In [10]:
# TODO: train with a lower lr (5e-5)
# test properly using test-ar.txt
# short sentences
# bf16 if locally
# revisit Pointer Networks in case nlp handles EVERY single thing

In [1]:
import torch
import random
import numpy as np
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, TrainingArguments,Trainer, AutoTokenizer, DataCollatorForSeq2Seq, T5TokenizerFast

In [2]:
def scramble_sentence(sentence, shuffle_prob=0.9, max_shuffle_window=5):
    tokens = sentence.split()

    if random.random() < shuffle_prob and len(tokens) > 2:
        new_tokens = tokens[:]
        for i in range(len(tokens)):
            if random.random() < 0.75:
                j = min(len(tokens)-1, i + random.randint(1, max_shuffle_window))
                new_tokens[i], new_tokens[j] = new_tokens[j], new_tokens[i]
        tokens = new_tokens
    return " ".join(tokens)

In [3]:
def build_dataset(sl_sentences, num_variants=5):
    inputs = []
    targets = []

    for sent in sl_sentences:
        for _ in range(num_variants):
            noisy = scramble_sentence(sent)
            inputs.append(noisy)
            targets.append(sent)

    return inputs, targets

In [ ]:
from datasets import Dataset, DatasetDict

def load_parallel_ds(sourcePath, targetPath):
  with open(sourcePath, 'r', encoding="utf-8") as f_src, open(targetPath, 'r', encoding="utf-8") as t_src:
    sources = [line.strip() for line in f_src.readlines() if line.strip()]
    targets = [line.strip() for line in t_src.readlines() if line.strip()]

    if len(sources) != len(targets):
      raise ValueError("Source and target files do not have the same number of lines")

    return Dataset.from_dict({
            "input_text": sources,
            "target_text": targets
        })

dataset = DatasetDict({
    "train": load_parallel_ds("/content/train-ar.txt", "/content/ArSL-tr01-glossed+Processed.txt"),
    "validation": load_parallel_ds("/content/validation-ar.txt", "/content/ArSL-val01-glossed+Processed.txt"),
    "test": load_parallel_ds("/content/test-ar.txt", "/content/ArSL-ts01-glossed+Processed.txt")
})

# there is another variant of each glossed+preprocessed file (02), but for now i will only be using (01) versions
# they contain the same sentences but with different ordering

In [5]:
dataset['train']['input_text']

Column(['مرض اسهال علامة', 'الان أزمة قوي أنا قلبية شعور', 'قبل التبرع ملوث السبب نشر دم المرض', 'طبيب عملية جراحية العظم عمل الان', 'في شديد انا ازمة قلبية الان'])

In [7]:
dataset['test']['input_text']

Column(['قد يسبب غثيان المخدر لك', 'شلل الدماغي الشلل نصفي سبب', 'الهضمي الفم الجهاز بداية من', 'طلب طبيب القلب تجهيز العمليات غرفة', 'يعتمد الهوائية الجهاز التنفسي على القصبة'])

In [8]:
MODEL_NAME = "UBC-NLP/AraT5-base"
#MODEL_NAME = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)
MAX_LEN = 64

def tokenize(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    labels = tokenizer(
        batch["target_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]]

    return model_inputs

tokenized = dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

Map:   0%|          | 0/8100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model = model)

training_args = TrainingArguments(
    output_dir="./sl_reorder",
    eval_strategy="steps",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_steps=500,
    logging_steps=30,
    save_steps=500,
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator = data_collator
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss,Validation Loss
30,188.249919,111.051712
60,135.822664,46.786831
90,90.630623,42.746605
120,80.868896,43.962345
150,73.498250,34.764668
180,62.116890,26.682817
210,48.448417,18.224134
240,33.627765,12.629538
270,24.912059,10.129188
300,20.008687,8.508821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1270, training_loss=23.344400495994748, metrics={'train_runtime': 2522.8369, 'train_samples_per_second': 16.053, 'train_steps_per_second': 0.503, 'total_flos': 4397637795840000.0, 'train_loss': 23.344400495994748, 'epoch': 5.0})

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(110080, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(110080, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo

In [33]:
def reorder_to_sl(sentence):
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=5
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(reorder_to_sl("فحص"))
# جمجمة الرضيع صغيرة : الطفل جمجمة صغير
# يعطى التلقيح لكل الاطفال : يجب أخذ التطعيم كل الأطفال

فحص القلب سليم


In [ ]:
import torchinfo
torchinfo.summary(model)

Layer (type:depth-idx)                                       Param #
T5ForConditionalGeneration                                   --
├─Embedding: 1-1                                             84,639,744
├─T5Stack: 1-2                                               84,639,744
│    └─Embedding: 2-1                                        (recursive)
│    └─ModuleList: 2-2                                       --
│    │    └─T5Block: 3-1                                     7,079,808
│    │    └─T5Block: 3-2                                     7,079,424
│    │    └─T5Block: 3-3                                     7,079,424
│    │    └─T5Block: 3-4                                     7,079,424
│    │    └─T5Block: 3-5                                     7,079,424
│    │    └─T5Block: 3-6                                     7,079,424
│    │    └─T5Block: 3-7                                     7,079,424
│    │    └─T5Block: 3-8                                     7,079,424
│    │    └─T5Bloc

In [ ]:
model.save_pretrained("/content/AraT5v2")
tokenizer.save_pretrained("/content/AraT5v2")

In [18]:
def generate_predictions(sentences, batch_size=16):
    model.eval()
    preds = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LEN,
                num_beams=5
            )

        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=True
        )
        preds.extend(decoded)

    return preds


In [ ]:
raw_test = dataset["test"]
test_inputs = raw_test["input_text"]
test_targets = raw_test["target_text"]

test_preds = generate_predictions(test_inputs)

In [ ]:
for i in range(len(test_inputs)):
    print("="*80)
    print(f"[{i}] INPUT (scrambled):")
    print(test_inputs[i])
    print("\nTARGET (gold SL order):")
    print(test_targets[i])
    print("\nPREDICTED:")
    print(test_preds[i])


In [ ]:
exact_match = np.mean([
    p.strip() == t.strip()
    for p, t in zip(test_preds, test_targets)
])

print(f"Exact Match Accuracy: {exact_match:.4f}")

In [ ]:
df = pd.DataFrame({
    "input_scrambled": test_inputs,
    "target_sl": test_targets,
    "predicted_sl": test_preds
})

df.to_csv("test_predictions.csv", index=False, encoding="utf-8-sig")
